# Notebook 1: Use MCTS to generate chemical compounds based on user requirements

This notebook demonstrates how to manipulate MCTS search parameters to generate molecules for a simple use-case with RDKit.

Set a seed for reproducible MCTS results.

In [1]:
import random
seed = 42
random.seed(seed)

In [2]:
import sys
sys.path.insert(1, '../../minervachem/mcts')
from tree import *
from tree_viz import *

## 1: Setting MCTS search parameters

### Chemistry-specific parameters

MCTS can be used to optimize for any quantifiable chemical property.

In this example, we search for and generate molecules that trend toward a target solubility value.

We are using RDKit's function ```Descriptors.MolLogP()``` to calculate logP, a measure of solubility.

Set a target value for logP as well as the maximum allowed distance from the target.

In [3]:
goal = -0.5 # target logP value
max_value1 = 20 # max allowed logP value

If there are other chemical properties, you'd like to consider in the search, they can be easily added to the MCTS state.

In this example, we also consider a target synethesizability score (SA score) in addition to a target logP value.

SA score is evaluated with RDKit's function ```SA_Score.sascorer.calculateScore()```.

In [4]:
sa_target = 0 # target synthesizability score
max_value2 = 10 # max allowed SA score value

Reward value will be calculated by a weighted average of distance from the target logP and SA score.

In this demonstration, we are representing molecules as SMILES strings. MCTS will be generating molecules by growing SMILES strings. You can set the "move pool" by designating the SMILES symbols MCTS is allowed to add to the string.

Note: ```\n``` is a terminal symbol.

In [5]:
# SMILES symbols that are choices for molecular generation
choices = ['C', 'O', '=', 'N', 'c', '1', 'S', 'P', 'F', '\n']

Set the desired number of levels of the tree search. In the context of molecule generation, this is also the number of SMILES symbols in the optimized molecule.

In [6]:
levels = 5

### MCTS-specific parameters

Now, you can set parameters that will influence MCTS's search behavior.

```num_sims``` is the computational limit for MCTS

```turn``` is a counter to track the number of levels of the tree search

```beamsize``` is the number of top k candidates to select; Our version of MCTS does not select the single best child, it will select the k best children.

```alpha``` is the boltzmann constant; to lessen the distance between large and small values of children and to increase the chances of a small value child to be selected for exploration, we apply a Boltzmann distribution to the raw values of the children.

In [7]:
num_sims = 10000 # number of simulations, default 10000; set smaller to test code

turn = levels + 4 # counter to track number of turns

beamsize = 10 # set the beamsize

alpha = 2 # boltzmann constant

## 2: Running MCTS

With all user inputs set, we can now run MCTS to generate a batch of optimized molecules. With a fast evaluator like RDKit, this should take <30 seconds.

In [ ]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*') # these lines are to silence RDKit warnings from invalid molecules

current_nodes = [Node(LogP(
    goal=goal, 
    sa_target=sa_target, 
    allchoices=choices,
    max_value1=max_value1,
    turn=turn))]
for l in range(levels):
    next_nodes=utcbeam(budget=num_sims,rootpop=current_nodes, beamsize=beamsize, alpha=alpha, scalar=0.4)
    for i in current_nodes:
        print(f"Level {l}")
        print(f"This is one of the current nodes: {i}")
        print(f"Num Children: {len(i.children)}")
        for j,c in enumerate(i.children):
            print(j,c)
    print("These are the best children:")
    for i in next_nodes:
        print(i)
    current_nodes = next_nodes
    print("--------------------------------")

We can display the graphical representations of the generated molecules.

In [ ]:
allsmiles = []
allmols = []

for i in current_nodes:
    smiles = i.state.smiles
    allsmiles += [smiles]

    mol = Chem.MolFromSmiles(smiles)
    allmols.append(mol)
    mol.SetProp("name", str(Chem.rdMolDescriptors.CalcMolFormula(mol)))
    mol.SetProp("logP", str(f"{Descriptors.MolLogP(mol):.3f}"))
    mol.SetProp("SA score", str(f"{sascorer.calculateScore(mol):.3f}"))

img = Chem.Draw.MolsToGridImage(
    allmols, 
    legends=[f"{mol.GetProp('name')}" for mol in allmols],
    subImgSize=(350,350))

img

## 3: Plot tree searches

To help understand MCTS's search behavior with the set parameters, we can plot each final molecule's tree. Each visualized tree shows node values of interest at every level. Options for plotting values are ```'visits'```, ```'reward'```, ```'logp'```, and ```'sa_score'```.

In [ ]:
for i, j in enumerate(current_nodes):
    tuples = make_tree_nodes(node=j, size=levels)
    node_info = make_node_info(node=j, size=levels)
    # display(node_info)
    mass_plotting(node_info=node_info, params=['reward', 'logp'], tuples=tuples, smiles=allsmiles[i])